In [27]:
import pandas as pd
from geopy.distance import geodesic
from scipy.spatial import cKDTree
import numpy as np

In [28]:
faults = pd.read_csv("../data/J1939Faults.csv")
faults.head(2)

/var/folders/h8/51mqc9v517x00_mhcb5s7ncw0000gn/T/ipykernel_57628/1510985123.py:1: DtypeWarning: Columns (15) have mixed types. Specify dtype option on import or set low_memory=False.
  faults = pd.read_csv("../data/J1939Faults.csv")


,RecordID,ESS_Id,EventTimeStamp,eventDescription,actionDescription,ecuSoftwareVersion,ecuSerialNumber,ecuModel,ecuMake,ecuSource,spn,fmi,active,activeTransitionCount,faultValue,EquipmentID,MCTNumber,Latitude,Longitude,LocationTimeStamp
0,1,990349,2015-02-21 10:47:13.000,Low (Severity Low) Engine Coolant Level,NaN,unknown,unknown,unknown,unknown,0,111,17,True,2,NaN,1439,105354361,38.857638,-84.626851,2015-02-21 11:34:25.000
1,2,990360,2015-02-21 11:34:34.000,NaN,NaN,unknown,unknown,unknown,unknown,11,629,12,True,127,NaN,1439,105354361,38.857638,-84.626851,2015-02-21 11:35:10.000


### Possible Features:
* RecordID is unique
* EventTimeStamp has 1,050,909 unique values (no nan)
* eventDescription has 60,845 nan 
* ecuSoftwareVersion has 1,899 unique values (296,050 nan)
* ecuModel has 30 unique values (64,758 nan)
* ecuMake has 23 unique values (64,758 nan)
* ecuSource has 5 unique values (no nan)
* spn has 450 unique values (no nan)
* fmi has 26 unique values (no nan)
* active has 2 unique values (no nan)
* EquipmentID has 1,927 unique values (no nan)
* MCTNumber has 768 unique values (no nan)
* Latitude has 211,823 unique values (no nan)
* Longitude has 265,211 unique values (no nan)
* LocationTimeStamp has 1,036,006 unique values (no nan)

### Drop Columns:
* ESS_Id
* actionDescription
* ecuSerialNumber
* faultValue

In [3]:
len(faults['LocationTimeStamp'].unique())

1036006

In [4]:
faults['LocationTimeStamp'].isna().sum()

np.int64(0)

In [5]:
# Convert timestamps to datetime objects
faults['EventTimeStamp'] = pd.to_datetime(faults['EventTimeStamp'])
faults['LocationTimeStamp'] = pd.to_datetime(faults['LocationTimeStamp'])

In [30]:
diagnostics = pd.read_csv("../data/VehicleDiagnosticOnboardData.csv")
diagnostics.head(2)

,Id,Name,Value,FaultId
0,1,IgnStatus,False,1
1,2,EngineOilPressure,0,1


### To get the on-board diagnostics at the time of the fault code, we can match the **RecordID** to the **FaultId**.

In [16]:
diagnostics.loc[diagnostics['FaultId'] == 1]

,Id,Name,Value,FaultId
0,1,IgnStatus,False,1
1,2,EngineOilPressure,0,1
2,3,EngineOilTemperature,96.74375,1
3,4,TurboBoostPressure,0,1
4,5,EngineLoad,11,1
5,6,AcceleratorPedal,0,1
6,7,IntakeManifoldTemperature,78.8,1
7,8,FuelRate,0,1
8,9,FuelLtd,12300.907429328,1
9,10,EngineRpm,0,1


This data is in long-format, so each FaultId can have potentially many diagnostic values.

**Note:** Not all diagnostic values are recorded for all faults, so you will have a large number of missing values.

For example, for the second fault code in our dataset, only the ignition status and lamp status were recorded.

In [8]:
diagnostics.loc[diagnostics['FaultId'] == 46]

,Id,Name,Value,FaultId
418,419,IgnStatus,True,46
419,420,LampStatus,22527,46


Finally, we can get a little bit more information about the different fault codes from the Service Fault Codes spreadsheet.

In [31]:
sfc = pd.read_excel("../data/Service Fault Codes_1_0_0_167.xlsx")
sfc.head(2)

/opt/anaconda3/lib/python3.13/site-packages/openpyxl/worksheet/_read_only.py:85: UserWarning: Data Validation extension is not supported and will be removed
  for idx, row in parser.parse():


,Published in CES 14602,Cummins Fault Code,Revision,PID,SID,MID,J1587 FMI,SPN,J1939 FMI,J2012 Pcode,Lamp Color,Lamp Device,Cummins Description,Algorithm Description
0,Y,111,167,Not Mapped,254,0,12,629,12,P0606,Red,Stop / Shutdown,Engine Control Module Critical Internal Failur...,Error internal to the ECM related to memory ha...
1,Y,112,167,Not Mapped,20,128,7,635,7,Not Mapped,Red,Stop / Shutdown,Engine Timing Actuator Driver Circuit - Mechan...,Mechanical failure in the engine timing actuat...


For a large number of fault codes, there are multiple records. For example, if we look at the rows for the first fault in our dataset, we see that there are two rows.

In [10]:
(
    sfc
    .loc[sfc['SPN'] == 5246]
    .loc[sfc['J1939 FMI'].isin([ 0, 15, 16, 19, 14])]
)

,Published in CES 14602,Cummins Fault Code,Revision,PID,SID,MID,J1587 FMI,SPN,J1939 FMI,J2012 Pcode,Lamp Color,Lamp Device,Cummins Description,Algorithm Description
2518,Y,3712,167,Not Mapped,Not Mapped,Not Mapped,0,5246,0,Not Mapped,Red,Stop / Shutdown,Aftertreatment SCR Operator Inducement - Data ...,SCR inducement of 5 mph derate - Fault Code 41...
2781,Y,4134,167,Not Mapped,Not Mapped,Not Mapped,0,5246,15,Not Mapped,Amber,Warning,Aftertreatment SCR Operator Inducement - Data ...,SCR inducement - Least Severe - Fault Code 371...
4338,Y,6254,167,Not Mapped,Not Mapped,Not Mapped,0,5246,16,Not Mapped,Amber,Warning,Aftertreatment SCR Operator Inducement Severit...,NaN


Or even more.

In [11]:
(
    sfc
    .loc[sfc['SPN'] == 629]
    .loc[sfc['J1939 FMI'] == 12]
)

,Published in CES 14602,Cummins Fault Code,Revision,PID,SID,MID,J1587 FMI,SPN,J1939 FMI,J2012 Pcode,Lamp Color,Lamp Device,Cummins Description,Algorithm Description
0,Y,111,167,Not Mapped,254,0,12,629,12,P0606,Red,Stop / Shutdown,Engine Control Module Critical Internal Failur...,Error internal to the ECM related to memory ha...
180,Y,343,167,Not Mapped,254,0,12,629,12,P0607,Amber,Warning,Engine Control Module Warning Internal Hardwar...,ECM power supply errors have been detected.
689,Y,1116,167,Not Mapped,254,0,12,629,12,Not Mapped,Amber,Warning,Engine Control Module Critical Internal Failur...,ECM Internal failure has occurred.
854,Y,1388,167,Not Mapped,254,0,12,629,12,Not Mapped,NaN,NaN,Engine Control Module Data Lost - Bad Intellig...,The ECM data has been lost.
1019,Y,1597,167,Not Mapped,254,0,12,629,12,Not Mapped,Maintenance,Maintenance,Engine Control Module Critical Internal Failur...,The ECM has occurred an internal failure.


In [12]:
faults[faults['spn'] == 5246]

,RecordID,ESS_Id,EventTimeStamp,eventDescription,actionDescription,ecuSoftwareVersion,ecuSerialNumber,ecuModel,ecuMake,ecuSource,spn,fmi,active,activeTransitionCount,faultValue,EquipmentID,MCTNumber,Latitude,Longitude,LocationTimeStamp
45,46,990931,2015-02-21 12:10:51,NaN,NaN,04993120*00027849*082113134117*07700053*I0*BBZ*,79464664,6X1u10D1500000000,CMMNS,0,5246,0,True,1,NaN,1395,105349612,36.065972,-86.433425,2015-02-21 12:11:27
1918,1919,1007751,2015-02-22 19:44:55,NaN,NaN,04993120*00027849*082113134117*07700053*I0*BBZ*,79464664,6X1u10D1500000000,CMMNS,0,5246,0,True,1,NaN,1395,105349612,36.066203,-86.434814,2015-02-22 19:46:27
2058,2059,1010486,2015-02-23 04:00:21,NaN,NaN,04993120*00027849*082113134117*07700053*I0*BBZ*,79464664,6X1u10D1500000000,CMMNS,0,5246,0,False,1,NaN,1395,105349612,36.066666,-86.434537,2015-02-23 01:06:06
2089,2090,1011009,2015-02-23 05:05:44,NaN,NaN,05290170*03015749*051914190353*09400015*G1*BDR*,79642446,6X1u13D1500000000,CMMNS,0,5246,0,True,1,NaN,1630,105329900,40.733009,-74.087777,2015-02-23 05:08:23
2971,2972,1026305,2015-02-23 15:54:22,NaN,NaN,unknown,unknown,unknown,unknown,0,5246,0,True,1,NaN,1487,105369355,28.077361,-81.897083,2015-02-23 15:54:58
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1183032,1244156,121610128,2020-02-19 07:02:33,NaN,NaN,05317106*05005224*051718172255*09401583*G1*BDR*,79845785,6X1u13D1500000000,CMMNS,0,5246,0,True,1,NaN,1814,105369518,36.067037,-86.434120,2020-02-19 07:03:09
1183684,1244808,121909497,2020-02-21 07:23:44,NaN,NaN,04384413*22246857*090619141107*60701756*G1*BGT*,80092582,6X1u17D1500000000,CMMNS,0,5246,16,True,1,NaN,2211,105329862,36.066296,-86.434305,2020-02-21 07:24:20
1184328,1245452,122305094,2020-02-24 15:28:05,NaN,NaN,04384413*22246857*090619141107*60701756*G1*BGT*,80092582,6X1u17D1500000000,CMMNS,0,5246,16,False,1,NaN,2211,105329862,36.066620,-86.434722,2020-02-24 15:28:01
1184330,1245454,122305096,2020-02-24 15:27:26,NaN,NaN,04384413*22246857*090619141107*60701756*G1*BGT*,80092582,6X1u17D1500000000,CMMNS,0,5246,0,True,1,NaN,2211,105329862,36.066620,-86.434722,2020-02-24 15:28:02


I know the readme says to remove the records near service locations, however, I think that can be done later. I don't think we need to remove the nan values just yet bc that could remove observations that could be useful.

I think the priority would be converting the timestamps and ordering the records chronologically, then linking the faults RecordIds with diagnostics FaultId. Maybe drop the id column and widen the diagnostics data.

* spn code 5246 has 1195 records with 5 unique fmi codes (no nan)

So, 1,195 records had full derates out of 1,187,335 records. Pretty imbalanced.

## Data Cleaning

### order the data chronologically

In [32]:
faults = faults.sort_values(by='EventTimeStamp', ascending=True)
faults.head(2)

,RecordID,ESS_Id,EventTimeStamp,eventDescription,actionDescription,ecuSoftwareVersion,ecuSerialNumber,ecuModel,ecuMake,ecuSource,spn,fmi,active,activeTransitionCount,faultValue,EquipmentID,MCTNumber,Latitude,Longitude,LocationTimeStamp
1154193,1211417,108604425,2000-03-18 19:14:10.000,High Voltage (Left Fuel Level Sensor),NaN,NaN,NaN,CECU3B-NAMUX4,PACCR,49,829,3,True,126,NaN,2015,105427130,36.935972,-86.507407,2000-03-18 19:14:46.000
1154194,1211418,108604426,2000-03-18 19:14:10.000,High Voltage (Fuel Level),NaN,NaN,NaN,CECU3B-NAMUX4,PACCR,49,96,3,True,126,NaN,2015,105427130,36.935972,-86.507407,2000-03-18 19:14:46.000


## service station

In [34]:
service_stations = [
    (36.0666667, -86.4347222),
    (35.5883333, -86.4438888),
    (36.1950, -83.174722)
]

In [38]:
from sklearn.neighbors import BallTree

def is_near_service_station_balltree(df, service_stations, threshold_distance=1.0):
    earth_radius_km = 6371.0
    
    stations_rad = np.radians(service_stations)
    points_rad = np.radians(df[['Latitude', 'Longitude']].values)
    
    tree = BallTree(stations_rad, metric='haversine')
    
    # Query radius in radians
    indices = tree.query_radius(points_rad, r=threshold_distance / earth_radius_km)
    
    is_near = np.array([len(idx) > 0 for idx in indices])
    
    return is_near

In [39]:
threshold_distance = 1.0 

# apply function
faults['IsServiceStation'] = is_near_service_station_balltree(
    faults, service_stations, threshold_distance)

In [40]:
faults['IsServiceStation'].value_counts(normalize = True)

IsServiceStation
False    0.88936
True     0.11064
Name: proportion, dtype: float64

### diagnostics features

In [41]:
diagnostics['Value'] = diagnostics['Value'].replace({'FALSE': 0, 'TRUE': 1})

In [45]:
diagnostics_wide = diagnostics.pivot(index='FaultId', columns='Name', values='Value')
features = diagnostics_wide.reset_index()
features.columns.name = None

## merge faults and feature

In [46]:
combined = pd.merge(faults, 
                    features, 
                    left_on='RecordID', 
                    right_on='FaultId',
                    how = 'left')

## what is a full derate?

In [47]:
combined['IsDerateFull'] = (combined['spn'] == 5246) & (combined['active'] == True)
combined['IsDerateFull'].value_counts()

IsDerateFull
False    1186728
True         607
Name: count, dtype: int64

In [48]:
faults['IsFullDerate'] = (faults['spn'] == 5246) & (faults['active'] == True)

## time cut off

In [24]:
cutoff_date = '2018-12-31 23:59:59'
training_faults_before_2019 = faults[faults['EventTimeStamp'] <= cutoff_date]
test_faults_after_2019 = faults[faults['EventTimeStamp'] > cutoff_date]

In [23]:
#test_faults_after_2019.to_csv('test.csv', index=False)
#training_faults_before_2019.to_csv('training.csv', index=False)

,RecordID,ESS_Id,EventTimeStamp,eventDescription,actionDescription,ecuSoftwareVersion,ecuSerialNumber,ecuModel,ecuMake,ecuSource,spn,fmi,active,activeTransitionCount,faultValue,EquipmentID,MCTNumber,Latitude,Longitude,LocationTimeStamp
1057599,1100434,72775650,2018-12-31 12:13:17,High Voltage (Left Fuel Level Sensor),NaN,NaN,NaN,CECU3B-NAMUX4,PACCR,49,829,3,True,126,NaN,1874,105340084,34.867685,-85.035000,2018-12-31 12:13:53
1057598,1100433,72775649,2018-12-31 12:13:17,High Voltage (Fuel Level),NaN,NaN,NaN,CECU3B-NAMUX4,PACCR,49,96,3,True,126,NaN,1874,105340084,34.867685,-85.035000,2018-12-31 12:13:53
1057600,1100435,72775693,2018-12-31 12:14:32,High (Severity Low) Water In Fuel Indicator,NaN,04358814*06074034*030816202706*09400153*G1*BDR*,79920189,6X1u13D1500000000,CMMNS,0,97,15,False,6,NaN,1952,105357932,39.783425,-77.713009,2018-12-31 12:14:28
1057601,1100436,72776965,2018-12-31 12:29:17,Low (Severity Low) Engine Coolant Level,NaN,05317106*05002234*050515205406*09400034*G1*BDR*,79845329,6X1u13D1500000000,CMMNS,0,111,17,True,126,NaN,1809,105411041,36.230370,-86.427916,2018-12-31 12:29:53
1057602,1100437,72777437,2018-12-31 12:35:12,Low (Severity Low) Engine Coolant Level,NaN,05317106*05002234*050515205406*09400034*G1*BDR*,79845329,6X1u13D1500000000,CMMNS,0,111,17,False,126,NaN,1809,105411041,36.287175,-86.443750,2018-12-31 12:35:08
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1187333,1248457,123906113,2020-03-06 14:14:13,Low (Severity Medium) Engine Coolant Level,NaN,04384413*22544852*090619141107*60701756*G1*BGT*,NaN,NaN,NaN,0,111,18,True,8,NaN,2377,108605700,35.030925,-85.321527,2020-03-06 14:14:49
1187334,1248458,123906131,2020-03-06 14:15:34,Low (Severity Medium) Engine Coolant Level,NaN,04384413*22544852*090619141107*60701756*G1*BGT*,NaN,NaN,NaN,0,111,18,False,8,NaN,2377,108605700,35.027314,-85.323472,2020-03-06 14:15:30
1113249,1161752,87903705,2026-05-16 14:44:11,Low Voltage (Catalyst Dosing Unit),NaN,unknown,unknown,unknown,unknown,0,3361,4,False,1,NaN,1744,105306493,35.586851,-86.444120,2019-05-23 07:44:25
1113250,1161753,87903706,2026-05-16 14:44:11,NaN,NaN,unknown,unknown,unknown,unknown,0,5742,4,False,1,NaN,1744,105306493,35.586851,-86.444120,2019-05-23 07:44:25


i need to make the training and test data into csv files